In [ ]:
import pandas as pd
from pathlib import Path

# Load the season game logs you listed
base_dir = Path("/Users/donma/Code/NBA-Analytics/data")
games_files = [
    base_dir / "games_2010-11_2026_05_28.csv",
    base_dir / "games_2011-12_2026_05_28.csv",
    base_dir / "games_2012-13_2026_05_28.csv",
    base_dir / "games_2013-14_2026_05_28.csv",
    base_dir / "games_2014-15_2026_05_28.csv",
    base_dir / "games_2015-16_2026_05_28.csv",
    base_dir / "games_2016-17_2026_05_28.csv",
    base_dir / "games_2017-18_2026_05_28.csv",
    base_dir / "games_2018-19_2026_05_28.csv",
    base_dir / "games_2019-20_2026_05_28.csv",
    base_dir / "games_2020-21_2026_05_28.csv",
    base_dir / "games_2021-22_2026_05_28.csv",
    base_dir / "games_2022-23_2026_05_28.csv",
    base_dir / "games_2023-24_2026_05_28.csv",
    base_dir / "games_2024-25_2026_05_28.csv",
    base_dir / "games_2025-26_2026_05_28.csv",
]

df_raw = pd.concat([pd.read_csv(path) for path in games_files], ignore_index=True)
df_raw["GAME_DATE"] = pd.to_datetime(df_raw["GAME_DATE"])
df_raw["season"] = df_raw["season"].astype(str)
df_raw["HOME"] = df_raw["MATCHUP"].str.contains(r"vs\.", regex=True).astype(int)
df_raw["TEAM_WIN"] = (df_raw["WL"] == "W").astype(int)
df_raw["eFG_PCT"] = (df_raw["FGM"] + 0.5 * df_raw["FG3M"]) / df_raw["FGA"].replace(0, pd.NA)
df_raw["AST_TO_RATIO"] = df_raw["AST"] / df_raw["TOV"].replace(0, pd.NA)

df_raw = df_raw.sort_values(["season", "TEAM_ABBREVIATION", "GAME_DATE", "GAME_ID"]).reset_index(drop=True)
team_group = df_raw.groupby(["season", "TEAM_ABBREVIATION"], group_keys=False)

df_raw["REST_DAYS"] = team_group["GAME_DATE"].transform(lambda series: series.diff().dt.days.sub(1))
df_raw["BACK_TO_BACK"] = (df_raw["REST_DAYS"] == 0).astype(int)
df_raw["ROLL_WIN_PCT_5"] = team_group["TEAM_WIN"].transform(lambda series: series.shift(1).rolling(5, min_periods=1).mean())
df_raw["ROLL_WIN_PCT_10"] = team_group["TEAM_WIN"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["TEAM_SEASON_WIN_PCT"] = team_group["TEAM_WIN"].transform(lambda series: series.shift(1).expanding(min_periods=1).mean())
df_raw["ROLL_FG_PCT_10"] = team_group["FG_PCT"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["ROLL_EFG_PCT_10"] = team_group["eFG_PCT"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["ROLL_AST_TO_RATIO_10"] = team_group["AST_TO_RATIO"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["ROLL_REB_10"] = team_group["REB"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["ROLL_STL_10"] = team_group["STL"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["ROLL_BLK_10"] = team_group["BLK"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())
df_raw["ROLL_TOV_10"] = team_group["TOV"].transform(lambda series: series.shift(1).rolling(10, min_periods=1).mean())

home_games = df_raw[df_raw["HOME"] == 1].copy()
away_games = df_raw[df_raw["HOME"] == 0].copy()

opponent_cols = [
    "GAME_ID",
    "season",
    "TEAM_ABBREVIATION",
    "MATCHUP",
    "TEAM_WIN",
    "ROLL_WIN_PCT_10",
    "ROLL_FG_PCT_10",
    "ROLL_REB_10",
    "ROLL_STL_10",
    "ROLL_BLK_10",
    "ROLL_TOV_10",
    "PTS",
    "FGM",
    "REB",
    "STL",
    "BLK",
    "TOV",
]

home_view = home_games.merge(
    away_games[opponent_cols],
    on=["GAME_ID", "season"],
    how="inner",
    suffixes=("", "_OPP"),
)
away_view = away_games.merge(
    home_games[opponent_cols],
    on=["GAME_ID", "season"],
    how="inner",
    suffixes=("", "_OPP"),
)

for frame in (home_view, away_view):
    frame["OPP_TEAM_ABBREVIATION"] = frame["TEAM_ABBREVIATION_OPP"]
    frame["TEAM_VS_OPP_SEASON_WIN_PCT"] = frame.groupby(
        ["season", "TEAM_ABBREVIATION", "OPP_TEAM_ABBREVIATION"]
    )["TEAM_WIN"].transform(lambda series: series.shift(1).expanding(min_periods=1).mean())
    frame["OPP_ROLL_WIN_PCT_10"] = frame["ROLL_WIN_PCT_10_OPP"]
    frame["OPP_ROLL_FG_PCT_10"] = frame["ROLL_FG_PCT_10_OPP"]
    frame["OPP_ROLL_REB_10"] = frame["ROLL_REB_10_OPP"]
    frame["OPP_ROLL_STL_10"] = frame["ROLL_STL_10_OPP"]
    frame["OPP_ROLL_BLK_10"] = frame["ROLL_BLK_10_OPP"]
    frame["OPP_ROLL_TOV_10"] = frame["ROLL_TOV_10_OPP"]
    frame["WIN"] = frame["TEAM_WIN"]

df = pd.concat([home_view, away_view], ignore_index=True)

pregame_feature_cols = [
    "HOME",
    "REST_DAYS",
    "BACK_TO_BACK",
    "ROLL_WIN_PCT_5",
    "ROLL_WIN_PCT_10",
    "ROLL_FG_PCT_10",
    "ROLL_EFG_PCT_10",
    "ROLL_AST_TO_RATIO_10",
    "ROLL_REB_10",
    "ROLL_STL_10",
    "ROLL_BLK_10",
    "ROLL_TOV_10",
    "OPP_ROLL_WIN_PCT_10",
    "OPP_ROLL_FG_PCT_10",
    "OPP_ROLL_REB_10",
    "OPP_ROLL_STL_10",
    "OPP_ROLL_BLK_10",
    "OPP_ROLL_TOV_10",
    "TEAM_VS_OPP_SEASON_WIN_PCT",
]

keep_cols = ["GAME_DATE", "season", "GAME_ID", "TEAM_ABBREVIATION", "MATCHUP", "WIN"] + pregame_feature_cols
df = df[keep_cols].sort_values("GAME_DATE").reset_index(drop=True)

rolling_feature_cols = [
    "ROLL_WIN_PCT_5",
    "ROLL_WIN_PCT_10",
    "ROLL_FG_PCT_10",
    "ROLL_EFG_PCT_10",
    "ROLL_AST_TO_RATIO_10",
    "ROLL_REB_10",
    "ROLL_STL_10",
    "ROLL_BLK_10",
    "ROLL_TOV_10",
    "OPP_ROLL_WIN_PCT_10",
    "OPP_ROLL_FG_PCT_10",
    "OPP_ROLL_REB_10",
    "OPP_ROLL_STL_10",
    "OPP_ROLL_BLK_10",
    "OPP_ROLL_TOV_10",
    "TEAM_VS_OPP_SEASON_WIN_PCT",
]

# Fill early-season missing rolling values with the season average for that feature.
# If a season average is still unavailable, fall back to 0.
for column_name in rolling_feature_cols:
    df[column_name] = df.groupby("season")[column_name].transform(lambda series: series.fillna(series.mean()))
    df[column_name] = df[column_name].fillna(0)




In [35]:
df['REST_DAYS'] = df['REST_DAYS'].fillna(df['REST_DAYS'].mean())
df[df['GAME_ID']=='0022500001']

,GAME_DATE,season,GAME_ID,TEAM_ABBREVIATION,MATCHUP,WIN,HOME,REST_DAYS,BACK_TO_BACK,ROLL_WIN_PCT_5,ROLL_WIN_PCT_10,ROLL_FG_PCT_10,ROLL_EFG_PCT_10,ROLL_AST_TO_RATIO_10,ROLL_REB_10,ROLL_STL_10,ROLL_BLK_10,ROLL_TOV_10,OPP_ROLL_WIN_PCT_10,OPP_ROLL_FG_PCT_10,OPP_ROLL_REB_10,OPP_ROLL_STL_10,OPP_ROLL_BLK_10,OPP_ROLL_TOV_10,TEAM_VS_OPP_SEASON_WIN_PCT


In [36]:
pd.set_option("display.max_columns", None)
df.describe()

,GAME_DATE,GAME_ID,WIN,HOME,REST_DAYS,BACK_TO_BACK,ROLL_WIN_PCT_5,ROLL_WIN_PCT_10,ROLL_FG_PCT_10,ROLL_EFG_PCT_10,ROLL_AST_TO_RATIO_10,ROLL_REB_10,ROLL_STL_10,ROLL_BLK_10,ROLL_TOV_10,OPP_ROLL_WIN_PCT_10,OPP_ROLL_FG_PCT_10,OPP_ROLL_REB_10,OPP_ROLL_STL_10,OPP_ROLL_BLK_10,OPP_ROLL_TOV_10,TEAM_VS_OPP_SEASON_WIN_PCT
count,40880,4.088000e+04,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000,40880.000000
mean,2018-08-14 14:27:22.896281856,2.305795e+07,0.500000,0.500000,1.212104,0.188283,0.506904,0.508736,0.461169,0.520956,1.920805,43.552286,7.713852,4.914640,13.617524,0.508736,0.461169,43.552286,7.713852,4.914640,13.617524,0.500000
min,2010-10-26 00:00:00,2.100000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.298000,0.326923,0.500000,25.000000,2.000000,0.000000,3.000000,0.000000,0.298000,25.000000,2.000000,0.000000,3.000000,0.000000
25%,2014-11-09 00:00:00,2.140043e+07,0.000000,0.000000,1.000000,0.000000,0.400000,0.400000,0.444800,0.497370,1.644923,41.500000,6.800000,4.100000,12.400000,0.400000,0.444800,41.500000,6.800000,4.100000,12.400000,0.500000
50%,2018-04-25 00:00:00,2.180062e+07,0.500000,0.500000,1.000000,0.000000,0.600000,0.500000,0.461100,0.521283,1.881299,43.500000,7.610655,4.800000,13.500000,0.500000,0.461100,43.500000,7.610655,4.800000,13.500000,0.500000
75%,2022-11-04 00:00:00,2.220113e+07,1.000000,1.000000,1.000000,0.000000,0.600000,0.700000,0.477600,0.544847,2.155281,45.500000,8.500000,5.600000,14.700000,0.700000,0.477600,45.500000,8.500000,5.600000,14.700000,0.500000
max,2026-05-26 00:00:00,4.250032e+07,1.000000,1.000000,145.000000,1.000000,1.000000,1.000000,0.586000,0.666667,11.000000,64.000000,18.000000,18.000000,26.000000,1.000000,0.586000,64.000000,18.000000,18.000000,26.000000,1.000000
std,NaN,4.956026e+06,0.500006,0.500006,3.429672,0.390943,0.261880,0.218938,0.025006,0.034635,0.404621,3.083164,1.327199,1.146452,1.757840,0.218938,0.025006,3.083164,1.327199,1.146452,1.757840,0.273165


In [37]:
import numpy as np

seed = 36
rng = np.random.default_rng(seed)

def split_season_games(season_frame: pd.DataFrame, random_seed: int):
    game_ids = season_frame["GAME_ID"].drop_duplicates().to_numpy()
    rng_local = np.random.default_rng(random_seed)
    rng_local.shuffle(game_ids)

    game_count = len(game_ids)
    train_end = int(game_count * 0.70)
    test_end = train_end + int(game_count * 0.20)

    train_ids = game_ids[:train_end]
    test_ids = game_ids[train_end:test_end]
    val_ids = game_ids[test_end:]

    train_df = season_frame[season_frame["GAME_ID"].isin(train_ids)].copy()
    test_df = season_frame[season_frame["GAME_ID"].isin(test_ids)].copy()
    val_df = season_frame[season_frame["GAME_ID"].isin(val_ids)].copy()

    return train_df, test_df, val_df

season_groups = []
for season_name, season_frame in df.groupby("season", sort=True):
    train_df, test_df, val_df = split_season_games(season_frame, seed)
    season_groups.append((train_df, test_df, val_df))

train_df = pd.concat([group[0] for group in season_groups], ignore_index=True).sort_values(["season", "GAME_DATE", "GAME_ID"]).reset_index(drop=True)
test_df = pd.concat([group[1] for group in season_groups], ignore_index=True).sort_values(["season", "GAME_DATE", "GAME_ID"]).reset_index(drop=True)
val_df = pd.concat([group[2] for group in season_groups], ignore_index=True).sort_values(["season", "GAME_DATE", "GAME_ID"]).reset_index(drop=True)

print(f"train rows: {len(train_df)}")
print(f"test rows: {len(test_df)}")
print(f"val rows: {len(val_df)}")
print(f"unique train games: {train_df['GAME_ID'].nunique()}")
print(f"unique test games: {test_df['GAME_ID'].nunique()}")
print(f"unique val games: {val_df['GAME_ID'].nunique()}")
print("rows per game in train/test/val:", train_df.groupby("GAME_ID").size().min(), test_df.groupby("GAME_ID").size().min(), val_df.groupby("GAME_ID").size().min())


train rows: 28600
test rows: 8160
val rows: 4120
unique train games: 14300
unique test games: 4080
unique val games: 2060
rows per game in train/test/val: 2 2 2


In [38]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss, roc_auc_score

from xgboost import XGBClassifier


X_train = train_df[pregame_feature_cols]
y_train = train_df["WIN"].astype(int)
X_test = test_df[pregame_feature_cols]
y_test = test_df["WIN"].astype(int)
X_val = val_df[pregame_feature_cols]
y_val = val_df["WIN"].astype(int)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric=['logloss', 'auc'],
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    random_state=seed,
    tree_method="hist",
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
test_preds = xgb_model.predict(X_test)
test_proba = xgb_model.predict_proba(X_test)[:, 1]
val_preds = xgb_model.predict(X_val)
val_proba = xgb_model.predict_proba(X_val)[:, 1]


print("Accuracy:", accuracy_score(y_val, val_preds))
print("Classification Report:\n", classification_report(y_val, val_preds))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_preds))
print("Log Loss:", log_loss(y_val, val_proba))
print("ROC AUC:", roc_auc_score(y_val, val_proba))

feature_importance = pd.Series(xgb_model.feature_importances_, index=pregame_feature_cols).sort_values(ascending=False)
feature_importance


Accuracy: 0.6140776699029126
Classification Report:
               precision    recall  f1-score   support

           0       0.61      0.61      0.61      2060
           1       0.61      0.61      0.61      2060

    accuracy                           0.61      4120
   macro avg       0.61      0.61      0.61      4120
weighted avg       0.61      0.61      0.61      4120

Confusion Matrix:
 [[1264  796]
 [ 794 1266]]
Log Loss: 0.6639288084357337
ROC AUC: 0.65724149307192


HOME                          0.183750
ROLL_WIN_PCT_10               0.095845
OPP_ROLL_WIN_PCT_10           0.086133
TEAM_VS_OPP_SEASON_WIN_PCT    0.054280
ROLL_EFG_PCT_10               0.044103
OPP_ROLL_FG_PCT_10            0.043965
ROLL_FG_PCT_10                0.043596
REST_DAYS                     0.042755
OPP_ROLL_TOV_10               0.042133
ROLL_AST_TO_RATIO_10          0.041876
ROLL_TOV_10                   0.041013
OPP_ROLL_REB_10               0.040998
ROLL_REB_10                   0.040842
ROLL_STL_10                   0.040781
OPP_ROLL_STL_10               0.040184
OPP_ROLL_BLK_10               0.039640
ROLL_BLK_10                   0.039581
ROLL_WIN_PCT_5                0.038523
BACK_TO_BACK                  0.000000
dtype: float32

In [39]:
import optuna
from optuna.samplers import TPESampler

def hyper_finetune(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": ["logloss", "auc"],
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": seed,
        "tree_method": "hist",
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_proba = model.predict_proba(X_val)[:, 1]
    return log_loss(y_val, val_proba)

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=seed))
study.optimize(hyper_finetune, n_trials=50)

best_params = study.best_params
best_value = study.best_value
print("Best Log Loss:", best_value)
print("Best Hyperparameters:", best_params)

[I 2026-05-29 15:09:33,778] A new study created in memory with name: no-name-dce35ecd-9d58-4919-aeec-9af3fa39612d
[I 2026-05-29 15:09:36,193] Trial 0 finished with value: 0.6976805056233967 and parameters: {'n_estimators': 756, 'learning_rate': 0.03995896330522187, 'max_depth': 10, 'min_child_weight': 2, 'subsample': 0.9036795649350824, 'colsample_bytree': 0.6816072982605145, 'reg_alpha': 4.775417104143057e-07, 'reg_lambda': 2.149617280075606e-06}. Best is trial 0 with value: 0.6976805056233967.
[I 2026-05-29 15:09:38,182] Trial 1 finished with value: 0.6466368546557161 and parameters: {'n_estimators': 667, 'learning_rate': 0.0107785644158755, 'max_depth': 10, 'min_child_weight': 10, 'subsample': 0.7750650778109162, 'colsample_bytree': 0.5985041438203735, 'reg_alpha': 1.2254496820421623e-06, 'reg_lambda': 1.050252624302251e-08}. Best is trial 1 with value: 0.6466368546557161.
[I 2026-05-29 15:09:38,772] Trial 2 finished with value: 0.6428927471615201 and parameters: {'n_estimators': 30

Best Log Loss: 0.6412997129478153
Best Hyperparameters: {'n_estimators': 846, 'learning_rate': 0.010343345183195804, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.6588348850024701, 'colsample_bytree': 0.9451376024110426, 'reg_alpha': 0.00040855951557390824, 'reg_lambda': 9.432862749943794}


In [40]:
best_xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric=["logloss", "auc"],
    random_state=seed,
    tree_method="hist",
    **best_params,
)

X_train_full = pd.concat([train_df, val_df], ignore_index=True)[pregame_feature_cols]
y_train_full = pd.concat([train_df, val_df], ignore_index=True)["WIN"].astype(int)

best_xgb_model.fit(X_train_full, y_train_full, verbose=False)

test_proba_best = best_xgb_model.predict_proba(X_test)[:, 1]
test_preds_best = best_xgb_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, test_preds_best))
print("Test Log Loss:", log_loss(y_test, test_proba_best))
print("Test ROC AUC:", roc_auc_score(y_test, test_proba_best))

feature_importance = pd.Series(best_xgb_model.feature_importances_, index=pregame_feature_cols).sort_values(ascending=False)
feature_importance

Test Accuracy: 0.6365196078431372
Test Log Loss: 0.6399033856586931
Test ROC AUC: 0.6837396674356017


HOME                          0.181580
OPP_ROLL_WIN_PCT_10           0.144500
ROLL_WIN_PCT_10               0.142265
ROLL_WIN_PCT_5                0.070692
TEAM_VS_OPP_SEASON_WIN_PCT    0.057098
BACK_TO_BACK                  0.040996
OPP_ROLL_FG_PCT_10            0.038174
ROLL_EFG_PCT_10               0.037431
REST_DAYS                     0.035549
ROLL_FG_PCT_10                0.030593
OPP_ROLL_TOV_10               0.028615
ROLL_TOV_10                   0.027048
ROLL_AST_TO_RATIO_10          0.026050
OPP_ROLL_REB_10               0.024060
OPP_ROLL_STL_10               0.023429
OPP_ROLL_BLK_10               0.023244
ROLL_STL_10                   0.022940
ROLL_BLK_10                   0.022893
ROLL_REB_10                   0.022842
dtype: float32